In [ ]:
import os
import sys

# Make src/ importable
project_root = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(project_root, "src"))

# 1. Import necessary modules
from classification_model import data_load, features, train, evaluation
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 2. Load the raw data
df = data_load.load_data()
print(f"Raw data shape: {df.shape}")
display(df.head(5))  # Show first 5 rows of the dataset

# 3. Preprocess the data
X, y = features.preprocess_data(df)
print(f"Processed feature matrix shape: {X.shape}")
print(f"Churn rate in full dataset: {y.mean():.2%} positive")

# 4. Split into train/test sets for modeling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

# 5. Handle class imbalance on training data (oversample churn=Yes)
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print(f"After SMOTE oversampling, training size: {X_train_bal.shape[0]} (Churn=1 count: {sum(y_train_bal==1)})")

# 6. Train the model (using XGBoost in this example)
model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
model.fit(X_train_bal, y_train_bal)

# 7. Evaluate the model on test set
y_pred = model.predict(X_test)
evaluation.evaluate_model(y_test, y_pred)

# 8. Plot ROC curve
y_probs = model.predict_proba(X_test)[:, 1]
evaluation.plot_roc_curve(y_test, y_probs)


In [ ]:
# Connecting to classification_model/train.py to test different models

import os
import sys

project_root = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(project_root, "src"))

from classification_model import train

model, X_test, y_test= train.train_and_evaluate(model_name="logistic")
print(model)
print("-------------------")
model, X_test, y_test= train.train_and_evaluate(model_name="xgb")
print(model)
print("-------------------")
model, X_test, y_test= train.train_and_evaluate(model_name="lgbm")
print(model)
print("-------------------")
model, X_test, y_test= train.train_and_evaluate(model_name="rf")
print(model)
